# 04 — Current World Cup 2026 odds ledger

This notebook can be run repeatedly before and during the tournament.

Input:
- The Odds API key.

Output:
- current odds snapshot;
- accumulated odds ledger;
- odds movement report;
- `data/processed/04_current_odds_ledger_report_bundle.zip`.

Credits:
- `eu` + `h2h` = about 1 credit per run;
- `eu,uk,us` + `h2h` = about 3 credits per run.


In [ ]:
# Cell 1. Paste The Odds API key here.

ODDS_API_KEY = "PASTE_THE_ODDS_API_KEY_HERE"


In [ ]:
# Cell 2. Imports and helpers.


from pathlib import Path
import json
import re
import zipfile
import time

import numpy as np
import pandas as pd

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = pd.Timestamp.utcnow().isoformat()
SEED = 42

TEAM_REPLACEMENTS = {
    "USA": "United States",
    "US": "United States",
    "Korea Republic": "South Korea",
    "Türkiye": "Turkey",
    "Czechia": "Czech Republic",
    "Bosnia and Herzegovina": "Bosnia & Herzegovina",
    "Ivory Coast": "Côte d'Ivoire",
    "Cote d Ivoire": "Côte d'Ivoire",
}

def norm_team(name):
    if pd.isna(name):
        return ""
    name = str(name).strip()
    name = re.sub(r"\s+", " ", name)
    return TEAM_REPLACEMENTS.get(name, name)

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            payload,
            f,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
    return path

def infer_outcome(home_goals, away_goals):
    if pd.isna(home_goals) or pd.isna(away_goals):
        return np.nan
    if home_goals > away_goals:
        return 2
    if home_goals < away_goals:
        return 0
    return 1

def expected_score(rating_a, rating_b):
    return 1.0 / (1.0 + 10 ** (-(rating_a - rating_b) / 400.0))



import requests

ODDS_BASE = "https://api.the-odds-api.com/v4"

def has_key(value):
    if value is None:
        return False
    value = str(value).strip()
    if value == "":
        return False
    if value.startswith("PASTE_"):
        return False
    return True

def mask_key(value):
    if not has_key(value):
        return "missing"
    value = str(value)
    return f"present_len_{len(value)}_last4_{value[-4:]}"

def safe_get_json(url, params=None, timeout=60):
    try:
        response = requests.get(
            url,
            params=params or {},
            timeout=timeout,
        )
        try:
            data = response.json()
        except Exception:
            data = {"raw_text": response.text[:3000]}
        return response.status_code, data, dict(response.headers)
    except Exception as exc:
        return -1, {"error": str(exc)}, {}

def sanitize_params(params):
    out = dict(params or {})
    for key in list(out.keys()):
        if str(key).lower() in {"apikey", "api_key", "key", "token"}:
            out[key] = mask_key(out[key])
    return out

def usage_headers(headers):
    return {
        "x_requests_remaining": headers.get("x-requests-remaining"),
        "x_requests_used": headers.get("x-requests-used"),
        "x_requests_last": headers.get("x-requests-last"),
    }

def add_report_row(
    rows,
    check,
    endpoint,
    params,
    status,
    data,
    headers,
    path,
):
    row_count = 0
    if isinstance(data, list):
        row_count = len(data)
    elif isinstance(data, dict):
        value = data.get("data")
        if isinstance(value, list):
            row_count = len(value)
    error = ""
    if isinstance(data, dict):
        error = str(data.get("message", data.get("error", "")))[:500]
    rows.append({
        "run_timestamp_utc": RUN_TIMESTAMP_UTC,
        "check": check,
        "endpoint": endpoint,
        "params_masked": json.dumps(sanitize_params(params)),
        "status_code": status,
        "rows": row_count,
        "error": error,
        "cache_path": str(path),
        **usage_headers(headers),
    })

def odds_get(
    path,
    params=None,
    name="request",
    out_dir=None,
    report_rows=None,
):
    params = dict(params or {})
    params["apiKey"] = ODDS_API_KEY
    url = f"{ODDS_BASE}/{path.lstrip('/')}"
    status, data, headers = safe_get_json(url, params=params)
    safe_name = re.sub(r"[^a-zA-Z0-9_]+", "_", name)
    if out_dir is not None:
        out_path = Path(out_dir) / "raw" / f"{safe_name}.json"
        save_json(out_path, {
            "url_path": path,
            "params_masked": sanitize_params(params),
            "status_code": status,
            "headers": headers,
            "data": data,
        })
    else:
        out_path = Path(f"{safe_name}.json")
    if report_rows is not None:
        add_report_row(
            report_rows,
            check=name,
            endpoint=path,
            params=params,
            status=status,
            data=data,
            headers=headers,
            path=out_path,
        )
    return status, data, headers


LEDGER_DIR = PROCESSED_DIR / "current_wc2026_odds_ledger"
LEDGER_DIR.mkdir(parents=True, exist_ok=True)

SPORT_KEY = "soccer_fifa_world_cup"
REGIONS = "eu"
MARKETS = "h2h"

if not has_key(ODDS_API_KEY):
    raise ValueError("Paste ODDS_API_KEY in Cell 1 first.")

print("Setup OK:", ODDS_BASE)


In [ ]:
# Cell 3. Fetch current odds.

report_rows = []

status, data, headers = odds_get(
    f"sports/{SPORT_KEY}/odds",
    params={
        "regions": REGIONS,
        "markets": MARKETS,
        "oddsFormat": "decimal",
        "dateFormat": "iso",
    },
    name="current_wc2026_h2h",
    out_dir=LEDGER_DIR,
    report_rows=report_rows,
)

print("Status:", status)
print("Usage:", usage_headers(headers))

if not isinstance(data, list):
    print(data)
    current_snapshot = pd.DataFrame()
else:
    rows = []

    for event in data:
        event_id = event.get("id")
        commence_time = event.get("commence_time")
        home_team = norm_team(event.get("home_team"))
        away_team = norm_team(event.get("away_team"))

        bm_rows = []

        for bookmaker in event.get("bookmakers", []) or []:
            bookmaker_key = bookmaker.get("key")
            bookmaker_title = bookmaker.get("title")

            for market in bookmaker.get("markets", []) or []:
                if market.get("key") != "h2h":
                    continue

                prices = {}

                for outcome in market.get("outcomes", []) or []:
                    name = norm_team(outcome.get("name"))
                    price = pd.to_numeric(
                        outcome.get("price"),
                        errors="coerce",
                    )

                    if pd.isna(price):
                        continue

                    if name == home_team:
                        prices["home_odds"] = float(price)
                    elif name == away_team:
                        prices["away_odds"] = float(price)
                    elif str(name).lower() in {"draw", "tie"}:
                        prices["draw_odds"] = float(price)

                if {"home_odds", "draw_odds", "away_odds"} <= set(prices):
                    bm_rows.append({
                        "bookmaker_key": bookmaker_key,
                        "bookmaker_title": bookmaker_title,
                        **prices,
                    })

        if not bm_rows:
            continue

        bm = pd.DataFrame(bm_rows)

        avg_home = bm["home_odds"].mean()
        avg_draw = bm["draw_odds"].mean()
        avg_away = bm["away_odds"].mean()

        raw = np.array([
            1.0 / avg_away,
            1.0 / avg_draw,
            1.0 / avg_home,
        ])

        probs = raw / raw.sum()

        rows.append({
            "run_timestamp_utc": RUN_TIMESTAMP_UTC,
            "event_id": event_id,
            "commence_time": commence_time,
            "home_team": home_team,
            "away_team": away_team,
            "n_bookmakers": bm["bookmaker_key"].nunique(),
            "avg_home_odds": float(avg_home),
            "avg_draw_odds": float(avg_draw),
            "avg_away_odds": float(avg_away),
            "best_home_odds": float(bm["home_odds"].max()),
            "best_draw_odds": float(bm["draw_odds"].max()),
            "best_away_odds": float(bm["away_odds"].max()),
            "market_margin_avg": float(raw.sum() - 1.0),
            "market_p_away_devig": float(probs[0]),
            "market_p_draw_devig": float(probs[1]),
            "market_p_home_devig": float(probs[2]),
        })

    current_snapshot = pd.DataFrame(rows)

safe_ts = RUN_TIMESTAMP_UTC.replace(":", "-")
snapshot_path = LEDGER_DIR / f"current_snapshot_{safe_ts}.csv"
current_snapshot.to_csv(snapshot_path, index=False)

ledger_path = LEDGER_DIR / "wc2026_current_odds_ledger.csv"

if ledger_path.exists():
    old = pd.read_csv(ledger_path)
    ledger = pd.concat([old, current_snapshot], ignore_index=True)
else:
    ledger = current_snapshot.copy()

if len(ledger) > 0:
    ledger = ledger.drop_duplicates(
        ["run_timestamp_utc", "event_id"],
        keep="last",
    )

ledger.to_csv(ledger_path, index=False)

display(current_snapshot.head(30))
print("snapshot rows:", len(current_snapshot))
print("ledger rows:", len(ledger))


In [ ]:
# Cell 4. Movement report.

ledger_path = LEDGER_DIR / "wc2026_current_odds_ledger.csv"

ledger = pd.read_csv(ledger_path) if ledger_path.exists() else pd.DataFrame()

movement_rows = []

if len(ledger) > 0:
    ledger["run_timestamp_utc"] = pd.to_datetime(
        ledger["run_timestamp_utc"],
        utc=True,
        errors="coerce",
    )
    ledger = ledger.sort_values(["event_id", "run_timestamp_utc"])

    for event_id, group in ledger.groupby("event_id"):
        first = group.iloc[0]
        latest = group.iloc[-1]

        row = {
            "event_id": event_id,
            "home_team": latest["home_team"],
            "away_team": latest["away_team"],
            "commence_time": latest["commence_time"],
            "first_snapshot_time": first["run_timestamp_utc"],
            "latest_snapshot_time": latest["run_timestamp_utc"],
            "n_snapshots": len(group),
        }

        for outcome in ["home", "draw", "away"]:
            b = f"best_{outcome}_odds"
            a = f"avg_{outcome}_odds"

            row[f"{outcome}_best_first"] = first[b]
            row[f"{outcome}_best_latest"] = latest[b]
            row[f"{outcome}_best_change"] = latest[b] - first[b]

            row[f"{outcome}_avg_first"] = first[a]
            row[f"{outcome}_avg_latest"] = latest[a]
            row[f"{outcome}_avg_change"] = latest[a] - first[a]

        movement_rows.append(row)

movement = pd.DataFrame(movement_rows)

movement.to_csv(
    LEDGER_DIR / "wc2026_odds_movement_report.csv",
    index=False,
)

display(movement.head(30))


In [ ]:
# Cell 5. Save current odds report.

status_df = pd.DataFrame(report_rows)

status_df.to_csv(
    LEDGER_DIR / "current_odds_status.csv",
    index=False,
)

summary = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "snapshot_rows": int(len(current_snapshot)),
    "ledger_rows": int(len(ledger)),
    "regions": REGIONS,
    "markets": MARKETS,
}

save_json(LEDGER_DIR / "summary.json", summary)

zip_path = PROCESSED_DIR / "04_current_odds_ledger_report_bundle.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in LEDGER_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(LEDGER_DIR))

display(status_df)
print("Created:", zip_path.resolve())
